# Lab 1 - POS tagging, parsing and Bag of Words

**Topic:** part-of-speech (POS) tagging, dependency and constituency parsing, Bag of Words (BoW), and the effect of changing word order. The same sentence is used throughout, plus `harvard.wav` (speech).

**What each section demonstrates**

- **Part A - POS tagging:** tokens and Penn Treebank tags (NLTK), from text and from speech.
- **Part B - Parsing:** spaCy dependency table and interactive tree, plus an NLTK constituency (chunk) tree.
- **Part C - Bag of Words:** binary and skip-gram vectors (`bow.py`), and a BoW built by hand (`code2.py`).
- **Part D - Written analysis:** what BoW keeps and loses, and why parsing adds information (`task_d_analysis.md`).
- **Part E - Word order changed:** POS, parse and BoW again on a reordered sentence (`HOC.py`, plus the reversed-order POS run from `pos_tagging.py`).
- **code.py pipeline:** tokenisation, POS, named-entity chunking, frequency BoW and text-to-speech from one paragraph.

**Kernel:** Speech Lab (.venv)

**Notes on running**

- The speech cells call Google STT, which needs internet. If recognition fails, `utils.py` returns the built-in Harvard-sentence text, so the cells still run offline. The cell prints which source was used.
- The original scripts write `dependency_tree.html`, `HOC_light_dependency_tree.html` and `output.mp3` into their working folder. This notebook runs them from `notebooks/lab1_outputs/` so the repo's existing output files are not overwritten.
- Browser windows are not opened. The generated HTML trees are shown inline.

In [1]:
import os, sys, contextlib, warnings, webbrowser, importlib.util
from pathlib import Path

sys.dont_write_bytecode = True      # do not add __pycache__ files to the repo
from html import escape
from IPython.display import HTML, Markdown, Audio, display

ROOT = Path.cwd().parent            # notebook lives in notebooks/
SOL = ROOT / "lab 1" / "solution" / "POS, parser and BOW" / "1) POS, parser and BoW"
assert SOL.exists(), SOL
sys.path.insert(0, str(SOL))        # makes utils, pos_tagging, parsing, bow, HOC importable

OUT = ROOT / "notebooks" / "lab1_outputs"   # scratch folder for generated HTML / MP3
OUT.mkdir(parents=True, exist_ok=True)
os.chdir(OUT)                       # scripts write their outputs to the working folder

HARVARD = SOL / "harvard.wav"
SENTENCE = "The inventor built an incredibly fast car."


@contextlib.contextmanager
def no_browser():
    '''Record webbrowser.open() calls instead of opening a browser window.'''
    opened = []
    real_open = webbrowser.open
    webbrowser.open = lambda url, *args, **kwargs: (opened.append(url), False)[1]
    try:
        yield opened
    finally:
        webbrowser.open = real_open


def show_html_file(path, height=520):
    '''Display a generated HTML file inline (sandboxed in an iframe via srcdoc).'''
    html_text = Path(path).read_text(encoding="utf-8")
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Consider using IPython.display.IFrame")
        display(HTML(
            f'<iframe srcdoc="{escape(html_text, quote=True)}" '
            f'width="100%" height="{height}" style="border:1px solid #ddd;"></iframe>'
        ))


print("Repo root:      ", ROOT)
print("Solution folder:", SOL)
print("Working folder: ", Path.cwd())

Repo root:       U:\VITc\speech lab
Solution folder: U:\VITc\speech lab\lab 1\solution\POS, parser and BOW\1) POS, parser and BoW
Working folder:  U:\VITc\speech lab\notebooks\lab1_outputs


## Part A - POS tagging (text)

`pos_tagging.py` tokenises the sentence with NLTK `word_tokenize`, then tags each token with the Penn Treebank tagset (`nltk.pos_tag`). For example, `NN` = noun, `VBD` = past-tense verb, `JJ` = adjective, `RB` = adverb, `DT` = determiner.
This is the command `python pos_tagging.py "The inventor built an incredibly fast car."`, run through the module's own function.

In [2]:
import pos_tagging
from utils import get_text_from_input

text = get_text_from_input(SENTENCE)    # raw string -> returned unchanged
pos_tagging.perform_pos_tagging(text, label="Original")

Text: The inventor built an incredibly fast car.

Tokens:
['The', 'inventor', 'built', 'an', 'incredibly', 'fast', 'car', '.']



Part-of-Speech Tags:
The             -> DT
inventor        -> NN
built           -> VBD
an              -> DT
incredibly      -> RB
fast            -> JJ
car             -> NN
.               -> .


[('The', 'DT'),
 ('inventor', 'NN'),
 ('built', 'VBD'),
 ('an', 'DT'),
 ('incredibly', 'RB'),
 ('fast', 'JJ'),
 ('car', 'NN'),
 ('.', '.')]

### Part A (speech) - the same POS pipeline on `harvard.wav`

`get_text_from_input` sends `harvard.wav` to Google STT, which needs internet. It has a 10 s timeout and falls back to the built-in Harvard-sentence text if recognition fails or the network is down. The cell is wrapped in try/except so the rest of the notebook still runs.

In [3]:
FALLBACK_PREFIX = "The stale smell"     # first words of the utils.py fallback text

def speech_source(transcript):
    return ("offline fallback text (utils.py)"
            if transcript.startswith(FALLBACK_PREFIX) else "Google STT")

try:
    speech_text = get_text_from_input(str(HARVARD))
    print("\nTranscript source:", speech_source(speech_text))
    pos_tagging.perform_pos_tagging(speech_text, label="Speech (harvard.wav)")
except Exception as e:
    print("Speech step skipped:", type(e).__name__, e)

Reading audio file: U:\VITc\speech lab\lab 1\solution\POS, parser and BOW\1) POS, parser and BoW\harvard.wav


Google STT timed out (Network Issue). Using offline fallback text.



Transcript source: offline fallback text (utils.py)
Text: The stale smell of old beer lingers. It takes heat to bring out the odor. A cold dip restores health and zest. A salt pickle tastes fine with ham. Tacos al pastor are my favorite. A zestful food is the hot cross bun.

Tokens:
['The', 'stale', 'smell', 'of', 'old', 'beer', 'lingers', '.', 'It', 'takes', 'heat', 'to', 'bring', 'out', 'the', 'odor', '.', 'A', 'cold', 'dip', 'restores', 'health', 'and', 'zest', '.', 'A', 'salt', 'pickle', 'tastes', 'fine', 'with', 'ham', '.', 'Tacos', 'al', 'pastor', 'are', 'my', 'favorite', '.', 'A', 'zestful', 'food', 'is', 'the', 'hot', 'cross', 'bun', '.']

Part-of-Speech Tags:
The             -> DT
stale           -> NN
smell           -> NN
of              -> IN
old             -> JJ
beer            -> NN
lingers         -> NNS
.               -> .
It              -> PRP
takes           -> VBZ
heat            -> NN
to              -> TO
bring           -> VB
out             -> RP
the         

## Part B - Dependency and constituency parsing

`parsing.py` does two things.

1. **Dependency parsing (spaCy `en_core_web_sm`):** each word gets a grammatical relation (`dep_`) to its head word. Examples: `nsubj` (subject), `ROOT` (main verb), `dobj` (direct object), `amod` (adjective modifier), `det` (determiner). The table is printed, and an interactive tree is generated as `dependency_tree.html`. The original script opens that file in a browser. Here the browser call is recorded and the tree is shown inline.
2. **Constituency parsing (NLTK `RegexpParser`):** a small chunk grammar groups words into NP, PP, VP and CLAUSE phrases. Each leaf is printed as `word(TAG)`.

The line "Opened dependency tree in browser." comes from the original script. Because the browser call is suppressed in this notebook, that line is misleading here: the tree is shown inline instead.

In [4]:
import parsing

with no_browser() as opened:
    parsing.perform_parsing(SENTENCE, label="Original")

print("\nBrowser open requests suppressed:", len(opened))
show_html_file(OUT / "dependency_tree.html")


Original
Text: The inventor built an incredibly fast car.

Dependency Information
------------------------------------------------------------
WORD           POS       DEP            HEAD           
The            DET       det            inventor       
inventor       NOUN      nsubj          built          
built          VERB      ROOT           built          
an             DET       det            car            
incredibly     ADV       advmod         fast           
fast           ADJ       amod           car            
car            NOUN      dobj           built          
.              PUNCT     punct          built          

Generating interactive dependency tree...
Building dependency tree...

Dependency tree generated successfully.

File:
U:\VITc\speech lab\notebooks\lab1_outputs\dependency_tree.html

Opened dependency tree in browser.

Constituency Tree
------------------------------------------------------------
                                  S                   

### Part B (speech) - dependency and constituency parsing of the transcript

Same as above on the `harvard.wav` transcript (Google STT, or the offline fallback text). The tree below is regenerated from the transcript.

In [5]:
try:
    with no_browser() as opened:
        parsing.perform_parsing(speech_text, label="Speech (harvard.wav)")
    print("\nBrowser open requests suppressed:", len(opened))
    show_html_file(OUT / "dependency_tree.html")
except NameError as e:
    print("Speech step skipped (transcript not available):", e)
except Exception as e:
    print("Speech step skipped:", type(e).__name__, e)


Speech (harvard.wav)
Text: The stale smell of old beer lingers. It takes heat to bring out the odor. A cold dip restores health and zest. A salt pickle tastes fine with ham. Tacos al pastor are my favorite. A zestful food is the hot cross bun.

Dependency Information
------------------------------------------------------------
WORD           POS       DEP            HEAD           
The            DET       det            smell          
stale          ADJ       amod           smell          
smell          NOUN      nsubj          lingers        
of             ADP       prep           smell          
old            ADJ       amod           beer           
beer           NOUN      pobj           of             
lingers        NOUN      ROOT           lingers        
.              PUNCT     punct          lingers        
It             PRON      nsubj          takes          
takes          VERB      ROOT           takes          
heat           NOUN      dobj           takes         

## Part C - Bag of Words

A Bag of Words turns each document into a vector of counts over a fixed vocabulary, with word order discarded.

- **Binary BoW** (`CountVectorizer(binary=True)`): 1 if the word occurs in the document, 0 otherwise.
- **Skip-gram BoW** (`bow.skipgram_analyzer`): the features are word pairs taken from the tokens with up to 2 words skipped between them (`skipgrams(tokens, n=2, k=2)`). Some local order information is kept.

`bow.py` holds the skip-gram analyzer and a `__main__` demo with five hard-coded documents. That demo is copied below so it can run inside the notebook. Only the analyzer is imported from the module.

In [6]:
import bow                                   # provides bow.skipgram_analyzer
from sklearn.feature_extraction.text import CountVectorizer

hardcoded_docs = [
    "the inventor built the car",
    "the car was fast",
    "he saw the fast car",
    "the inventor built a clock",
    "the fast clock was built by him"
]

print("\nBinary Bag of Words (Hardcoded Docs)")
vectorizer_bin = CountVectorizer(binary=True)
try:
    X_bin = vectorizer_bin.fit_transform(hardcoded_docs)
    print("Vocabulary:", list(vectorizer_bin.get_feature_names_out()))
    for i, vec in enumerate(X_bin.toarray(), 1):
        print(f"Doc {i}: {vec}")
except ValueError:
    print("Failed to process hardcoded documents (Binary).")

print("\nSkip-Gram Bag of Words (Hardcoded Docs)")
vectorizer_sg = CountVectorizer(analyzer=bow.skipgram_analyzer)
try:
    X_sg = vectorizer_sg.fit_transform(hardcoded_docs)
    print("Vocabulary:", list(vectorizer_sg.get_feature_names_out()))
    for i, vec in enumerate(X_sg.toarray(), 1):
        print(f"Doc {i}: {vec}")
except Exception as e:
    print(f"Failed to process hardcoded skip-grams: {e}")


Binary Bag of Words (Hardcoded Docs)
Vocabulary: ['built', 'by', 'car', 'clock', 'fast', 'he', 'him', 'inventor', 'saw', 'the', 'was']
Doc 1: [1 0 1 0 0 0 0 1 0 1 0]
Doc 2: [0 0 1 0 1 0 0 0 0 1 1]
Doc 3: [0 0 1 0 1 1 0 0 1 1 0]
Doc 4: [1 0 0 1 0 0 0 1 0 1 0]
Doc 5: [1 1 0 1 1 0 1 0 0 1 1]

Skip-Gram Bag of Words (Hardcoded Docs)
Vocabulary: ['a clock', 'built a', 'built by', 'built car', 'built clock', 'built him', 'built the', 'by him', 'car fast', 'car was', 'clock built', 'clock by', 'clock was', 'fast built', 'fast car', 'fast clock', 'fast was', 'he fast', 'he saw', 'he the', 'inventor a', 'inventor built', 'inventor car', 'inventor clock', 'inventor the', 'saw car', 'saw fast', 'saw the', 'the a', 'the built', 'the car', 'the clock', 'the fast', 'the inventor', 'the the', 'the was', 'was built', 'was by', 'was fast', 'was him']
Doc 1: [0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 1 1 0 0 1 1 0 0
 0 0 0]
Doc 2: [0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

### Part C - BoW built by hand (`code2.py`)

`code2.py` has no functions, only top-level code, so its logic is copied here unchanged. It builds a sorted vocabulary from the four documents, then counts each vocabulary word in each document.

In [7]:
docs = [
    "the inventor built the car",
    "the car was fast",
    "he saw the fast car",
    "the inventor built a clock"
]

vocabulary = []
for doc in docs:
    words = doc.split()
    for word in words:
        if word not in vocabulary:
            vocabulary.append(word)


vocabulary.sort()

print("The Bag:")
print(vocabulary)


for i, doc in enumerate(docs, 1):
    words = doc.split()
    vector = []

    for vocab_word in vocabulary:
        word_count = words.count(vocab_word)
        vector.append(word_count)

    print(f"Doc {i}: '{doc}'")
    print(f"Bag of words: {vector}\n")

The Bag:
['a', 'built', 'car', 'clock', 'fast', 'he', 'inventor', 'saw', 'the', 'was']
Doc 1: 'the inventor built the car'
Bag of words: [0, 1, 1, 0, 0, 0, 1, 0, 2, 0]

Doc 2: 'the car was fast'
Bag of words: [0, 0, 1, 0, 1, 0, 0, 0, 1, 1]

Doc 3: 'he saw the fast car'
Bag of words: [0, 0, 1, 0, 1, 1, 0, 1, 1, 0]

Doc 4: 'the inventor built a clock'
Bag of words: [1, 1, 0, 1, 0, 0, 1, 0, 1, 0]



## Part E - Word order changed

Task E asks what happens when the word order changes. Two views of this are shown.

1. **`pos_tagging.py` (reversed order):** `utils.modify_word_order` reverses all the words. The POS tags are recomputed on the reversed sentence.
2. **`HOC.py`:** `modify_sentence` keeps the first and last words in place and reverses the words in between. The script then runs POS tagging, dependency parsing, the light interactive tree and the binary BoW on the new sentence. Its command is `python HOC.py "The aeroplane flies in the blue sky"`.

In [8]:
from utils import modify_word_order

reversed_text = modify_word_order(SENTENCE)
print("Reversed sentence:", reversed_text)
pos_tagging.perform_pos_tagging(reversed_text, label="Modified (Reversed Order)")

Reversed sentence: car. fast incredibly an built inventor The
Text: car. fast incredibly an built inventor The

Tokens:
['car', '.', 'fast', 'incredibly', 'an', 'built', 'inventor', 'The']

Part-of-Speech Tags:
car             -> NN
.               -> .
fast            -> NN
incredibly      -> RB
an              -> DT
built           -> VBN
inventor        -> NN
The             -> DT


[('car', 'NN'),
 ('.', '.'),
 ('fast', 'NN'),
 ('incredibly', 'RB'),
 ('an', 'DT'),
 ('built', 'VBN'),
 ('inventor', 'NN'),
 ('The', 'DT')]

In [9]:
import HOC

sentence = "The aeroplane flies in the blue sky"
modified = HOC.modify_sentence(sentence)

print("=" * 65)
print("HOC - MODIFIED SENTENCE PIPELINE")
print("=" * 65)
print("Original sentence:", sentence)
print("Modified sentence:", modified)

with no_browser() as opened:
    HOC.perform_pos_tagging(modified)
    HOC.perform_dependency_parsing(modified)
    HOC.visualize_dependency_tree(modified, output_file=str(OUT / "HOC_light_dependency_tree.html"))
    vocab, vector = HOC.binary_bow(modified)

print("\nBrowser open requests suppressed:", len(opened))
show_html_file(OUT / "HOC_light_dependency_tree.html")

HOC - MODIFIED SENTENCE PIPELINE
Original sentence: The aeroplane flies in the blue sky
Modified sentence: The blue the in flies aeroplane sky

PART-OF-SPEECH TAGGING
Text: The blue the in flies aeroplane sky

WORD              POS         TAG         LEMMA             
-----------------------------------------------------------------
The               DET         DT          the               
blue              NOUN        NN          blue              
the               DET         DT          the               
in                ADJ         JJ          in                
flies             NOUN        NNS         fly               
aeroplane         NOUN        NN          aeroplane         
sky               NOUN        NN          sky               

DEPENDENCY PARSING
Text: The blue the in flies aeroplane sky

WORD              POS         DEP            HEAD              
-----------------------------------------------------------------
The               DET         det          

In [10]:
# Same bag of words, different order: compare the lowercase word multisets.
same_bag = sorted(sentence.lower().split()) == sorted(modified.lower().split())
print("Same words in both sentences:", same_bag)
print("Same sentence order:         ", sentence.lower().split() == modified.lower().split())

Same words in both sentences: True
Same sentence order:          False


## code.py - full pipeline from text

`code.py` reads a paragraph from stdin (`echo "..." | python code.py`) and runs `process_text_pipeline`:

1. tokenisation (`word_tokenize`) and token count
2. POS tags
3. named-entity chunking (`nltk.ne_chunk`)
4. frequency-based BoW (`FreqDist`, most frequent first)
5. text-to-speech with gTTS, saved as `output.mp3` (needs internet)

The script calls `nltk.download(...)` at import time and reads stdin only under `__main__`. Here the module is loaded by file path (the name `code` clashes with the standard-library module) and the pipeline function is called with a paragraph string instead of stdin.

In [11]:
spec = importlib.util.spec_from_file_location("lab1_code_pipeline", SOL / "code.py")
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)

paragraph = (
    "The inventor built an incredibly fast car. "
    "It reached speeds that surprised the engineers, who were proud of the design."
)
pipeline.process_text_pipeline(paragraph)

mp3 = OUT / "output.mp3"
if mp3.exists():
    print("\nSaved speech:", mp3)
    display(Audio(str(mp3)))

Tokenization
['The', 'inventor', 'built', 'an', 'incredibly', 'fast', 'car', '.', 'It', 'reached', 'speeds', 'that', 'surprised', 'the', 'engineers', ',', 'who', 'were', 'proud', 'of', 'the', 'design', '.']
Total number of tokens: 23

POS Tag
[('The', 'DT'), ('inventor', 'NN'), ('built', 'VBD'), ('an', 'DT'), ('incredibly', 'RB'), ('fast', 'JJ'), ('car', 'NN'), ('.', '.'), ('It', 'PRP'), ('reached', 'VBD'), ('speeds', 'NNS'), ('that', 'WDT'), ('surprised', 'VBD'), ('the', 'DT'), ('engineers', 'NNS'), (',', ','), ('who', 'WP'), ('were', 'VBD'), ('proud', 'NNS'), ('of', 'IN'), ('the', 'DT'), ('design', 'NN'), ('.', '.')]


Parsing
(S
  The/DT
  inventor/NN
  built/VBD
  an/DT
  incredibly/RB
  fast/JJ
  car/NN
  ./.
  It/PRP
  reached/VBD
  speeds/NNS
  that/WDT
  surprised/VBD
  the/DT
  engineers/NNS
  ,/,
  who/WP
  were/VBD
  proud/NNS
  of/IN
  the/DT
  design/NN
  ./.)

Bag Of Words
  '.': 2
  'the': 2
  'The': 1
  'inventor': 1
  'built': 1
  'an': 1
  'incredibly': 1
  'fast': 1
  'car': 1
  'It': 1
  'reached': 1
  'speeds': 1
  'that': 1
  'surprised': 1
  'engineers': 1
  ',': 1
  'who': 1
  'were': 1
  'proud': 1
  'of': 1
  'design': 1




Saved speech: U:\VITc\speech lab\notebooks\lab1_outputs\output.mp3


## Part D - Written analysis

The written answer is in `task_d_analysis.md`. It is rendered below directly from the repo file. In short:

- **What BoW keeps:** the vocabulary and how often each word occurs. This is enough for keyword-based topic or spam classification.
- **What BoW loses:** word order, grammar, subject-verb-object relations, negation and context. "The dog bit the man" and "The man bit the dog" get identical BoW vectors.
- **What parsing adds:** dependency parsing gives who did what to whom. Constituency parsing gives the phrase hierarchy. Parse trees change when word order breaks grammar, which BoW cannot detect.

In [12]:
display(Markdown((SOL / "task_d_analysis.md").read_text(encoding="utf-8")))

# Task D: Analysis and Interpretation

This report addresses the questions formulated in Task D based on the outputs and observations from Tasks A, B, C, and E.

## 1. Which information is preserved by BoW?

Bag of Words (BoW) preserves the **vocabulary** and the **frequency** of each word within the document or audio segment. It tells us exactly *which* words are present and *how often* they occur. This makes BoW highly effective for document classification tasks where the presence of certain keywords is a strong indicator of the topic (e.g., classifying an email as spam if words like "free" and "money" appear frequently).

## 2. Which linguistic information is lost by BoW?

BoW completely loses all **syntactic and structural information**, including:
- **Word Order:** As demonstrated by our Task E implementation, reversing the word order of a sentence produces the exact same BoW vector. 
- **Grammar and Syntax:** Relationships between words (subject-verb-object) are lost. "The dog bit the man" and "The man bit the dog" have identical BoW representations, despite having completely opposite meanings.
- **Context and Semantic Nuance:** Sarcasm, negations (e.g., "not good" vs "good"), and contextual meanings of words are discarded because each word is treated as an isolated, independent entity.

## 3. Why can parsing provide information that BoW cannot?

Parsing (both Dependency and Constituency) analyzes the grammatical structure of a sentence, allowing it to provide information that BoW cannot:
- **Dependency Parsing** reveals the exact relationships between words (e.g., identifying which noun is the subject of the verb, and which adjectives modify which nouns). It tells us *who* did *what* to *whom*. 
- **Constituency Parsing** groups words into logical phrases (noun phrases, verb phrases), which helps in understanding the hierarchical structure of the sentence. 

By taking structural hierarchy and grammar rules into account, parsing can distinguish between sentences with identical vocabularies but different meanings (e.g., "The man bit the dog"), which BoW treats identically. Furthermore, as shown when we reorder words in Task E, the parse tree changes dramatically or breaks entirely, proving that parsing is sensitive to the word order and grammatical coherence that BoW ignores.


## Notes / what to look for

- **Part A:** each token gets one Penn Treebank tag. Check that "inventor" and "car" are nouns (NN), "built" is a past-tense verb (VBD), and "incredibly" is an adverb (RB) modifying "fast". The reversed-order run gives different tags for the same words, because the tagger uses neighbouring words as context.
- **Speech (A and B):** with internet, the transcript comes from Google STT. Offline, the cell reports the fallback source, and the results are the Harvard-sentence text. Both transcripts go through the same POS and parsing code.
- **Part B:** the dependency table has one `ROOT` (the main verb, "built"). Its `nsubj` child is "inventor" (subject) and its `dobj` child is "car" (object). The constituency chunker gives NP chunks "The inventor", "an" and "fast car". "built" and "incredibly" stay at the top level, because the simple grammar needs a noun phrase directly after the verb and no VP forms. This shows how limited a regex chunk grammar is compared with spaCy.
- **Multi-sentence input:** the speech transcript parses as one tree in spaCy, rooted at "are", so the HTML draws the whole transcript. In general `generate_dependency_tree` draws only `roots[0]` when spaCy finds several roots. The table still lists every token.
- **Part C:** binary vectors show presence only (0/1). Skip-gram features are word pairs with up to 2 skipped words, such as "the inventor" or "built car". Default `CountVectorizer` drops one-letter tokens, so "a" is absent from the binary vocabulary but present in the hand-built BoW of `code2.py`.
- **Part E:** the reversed and HOC sentences contain the same words, so the binary BoW of the modified sentence matches the original. The POS tags and parse change, though. In the HOC version spaCy tags "in" as ADJ and "flies" as NOUN, and makes "flies" the ROOT. This is the limitation discussed in Part D.
- **code.py:** the paragraph has no named entities, so `ne_chunk` returns a flat tree. The frequency BoW puts punctuation ("." appears twice) with the words. With internet, `output.mp3` is written to `notebooks/lab1_outputs/` and played above.